# MWIS QUBO, Annealing, and Reconstruction

This self-contained MWIS notebook reads the saved terminal subgraphs and decomposition map. It builds their QUBOs, runs D-Wave's classical simulated annealer, and combines the terminal choices and any explicit heuristic fallbacks into one schedule. It does not rebuild the conflict graph or decomposition.

This run is a classical simulation, not a quantum-hardware run. The saved QUBOs can be used separately with a real QPU when hardware access and credentials are available.

In [ ]:
from __future__ import annotations

import gzip
import json
import math
import os
from itertools import combinations
from pathlib import Path
from typing import Any

import networkx as nx
import pandas as pd

def find_project_root(start: Path | None = None) -> Path:
    """
    Find the repository root so paths work without further trouble.
    """
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").exists() and (candidate / "results").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing both `data/` and `results/`.")

# loading project root dir
PROJECT_ROOT = find_project_root()
RESULTS_DIR = PROJECT_ROOT / "results"


# --------- loading configuration environment ----------
# load .json config file (same file used by the MWIS pipeline)
CONFIG_PATH = Path(os.environ.get("SWE2_CONFIG", RESULTS_DIR / "_config.json"))
run_config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))

# input: any stage that produced terminal subgraphs (MWIS decomposition, CSP showcase, ...)
INPUT_RUN_DIR = RESULTS_DIR / run_config["qubo_input_run"]
PRODUCER_CONFIG_PATH = INPUT_RUN_DIR / "run_config_log.json"
producer_config = json.loads(PRODUCER_CONFIG_PATH.read_text(encoding="utf-8")) if PRODUCER_CONFIG_PATH.exists() else {}
TERMINAL_SUBGRAPHS_JSON = INPUT_RUN_DIR / "terminal_subgraphs.json"
if not TERMINAL_SUBGRAPHS_JSON.exists():
    raise FileNotFoundError(f"No terminal_subgraphs.json in {INPUT_RUN_DIR}. Run the producing stage first.")

# output: named after the input run, so QUBOs from different producers stay side by side
OUTPUT_RUN_DIR = RESULTS_DIR / "runs" / "mwis" / "qubo_annealing" / INPUT_RUN_DIR.name
OUTPUT_RUN_DIR.mkdir(parents=True, exist_ok=True)

# QUBO coefficients: scale the reward (linear) and the conflict penalty (quadratic).
# The penalty is computed from the effective reward scale, so changing the reward factor
# cannot accidentally make invalid conflicting selections attractive.
REWARD_FACTOR = float(run_config["qubo_reward_factor"])
PENALTY_FACTOR = float(run_config["qubo_penalty_factor"])
if not math.isfinite(REWARD_FACTOR) or REWARD_FACTOR <= 0:
    raise ValueError("qubo_reward_factor must be a positive finite number.")
if not math.isfinite(PENALTY_FACTOR) or PENALTY_FACTOR < 1.0:
    raise ValueError("qubo_penalty_factor must be a finite number >= 1.0 to preserve the MWIS penalty guarantee.")

# simulated-annealing settings
RUN_SIMULATED_ANNEALING = True
NUM_READS = run_config["num_reads"]
BASE_SEED = run_config["base_seed"]

# Log the selected producer settings with the annealing settings actually used.
run_config_log = {
    "producer_run": INPUT_RUN_DIR.name,
    "candidate_slot_csv": producer_config.get("candidate_slot_csv"),
    "max_qpu_terminal_nodes": producer_config.get("max_qpu_terminal_nodes"),
    "max_qpu_terminal_couplers": producer_config.get("max_qpu_terminal_couplers"),
    "exact_enumeration_node_limit": producer_config.get("exact_enumeration_node_limit"),
    "exact_clique_node_limit": producer_config.get("exact_clique_node_limit"),
    "qubo_input_run": run_config["qubo_input_run"],
    "qubo_reward_factor": REWARD_FACTOR,
    "qubo_penalty_factor": PENALTY_FACTOR,
    "num_reads": NUM_READS,
    "base_seed": BASE_SEED,
}
(OUTPUT_RUN_DIR / "run_config_log.json").write_text(json.dumps(run_config_log, indent=2), encoding="utf-8")

# terminal subgraphs produced by the decomposition stage
terminal_subgraphs = json.loads(TERMINAL_SUBGRAPHS_JSON.read_text(encoding="utf-8"))

print(f"Loaded configuration from: {CONFIG_PATH.resolve()}")
print(f"Loaded terminal subgraphs from: {INPUT_RUN_DIR.resolve()}")
print(f"Artifacts will be saved to: {OUTPUT_RUN_DIR.resolve()}")
print({"reward_factor": REWARD_FACTOR, "penalty_factor": PENALTY_FACTOR, "penalty_rule": "penalty_factor * (max effective reward + 1)", "num_reads": NUM_READS})
print({"terminal_subgraphs": len(terminal_subgraphs)})

## Build terminal QUBOs

In [ ]:
def graph_from_terminal(record: dict[str, Any]) -> nx.Graph:
    """
    Rebuild one terminal subgraph from its JSON record.
    """
    graph = nx.Graph()
    # re-building nodes
    for node in record["graph"]["nodes"]:
        node = dict(node)
        node_id = node.pop("id")          # "id" is the key, the rest are attributes (weight, times, ...)
        graph.add_node(node_id, **node)
    # re-building edges 
    for edge in record["graph"]["edges"]:
        edge = dict(edge)
        source = edge.pop("source")
        target = edge.pop("target")
        graph.add_edge(source, target, **edge)   # keeps the conflict "reason" attribute
    return graph


def max_effective_reward(graph: nx.Graph) -> float:
    """
    Largest scaled node reward in this terminal graph.
    """
    rewards = [REWARD_FACTOR * float(data["weight"]) for _, data in graph.nodes(data=True)]
    return max([0.0, *rewards])


def default_penalty(graph: nx.Graph) -> float:
    """
    Compute the conflict penalty above the largest effective reward.
    """
    return PENALTY_FACTOR * (max_effective_reward(graph) + 1.0)


def build_qubo_from_graph(graph: nx.Graph, penalty: float | None = None) -> dict[str, Any]:
    """
    Build QUBO instance startng from the given graph.
    """
    if penalty is None:
        penalty = default_penalty(graph)

    # linear reward
    linear = {node: -REWARD_FACTOR * float(data["weight"]) for node, data in graph.nodes(data=True)}

    # quadratic penalty
    quadratic = {(u, v): float(penalty) for u, v in graph.edges}

    return {
        "variables": sorted(graph.nodes),
        "linear": linear,
        "quadratic": quadratic,
        "penalty": float(penalty),
        "max_effective_reward": max_effective_reward(graph),
    }


def qubo_energy(qubo: dict[str, Any], sample: dict[str, int]) -> float:
    """
    Compute the energy of the QUBO assignment.
    """
    energy = sum(coef * sample.get(node, 0) for node, coef in qubo["linear"].items())
    energy += sum(coef * sample.get(u, 0) * sample.get(v, 0) for (u, v), coef in qubo["quadratic"].items())
    return energy


qubo_records = []   # for each temrinal graph build a separate QUBO formulation
term_rows = []      # flat list of every QUBO coefficient
for record in terminal_subgraphs:
    graph = graph_from_terminal(record)
    qubo = build_qubo_from_graph(graph)
    qubo_records.append({"terminal": record, "graph": graph, "qubo": qubo})

    for node, coef in qubo["linear"].items():
        term_rows.append({"terminal_id": record["terminal_id"], "term_type": "linear", "u": node, "v": "", "coefficient": coef})
    for (u, v), coef in qubo["quadratic"].items():
        term_rows.append({"terminal_id": record["terminal_id"], "term_type": "quadratic", "u": u, "v": v, "coefficient": coef})

# print out translation summary
qubo_summary = pd.DataFrame([
    {
        "terminal_id": item["terminal"]["terminal_id"],
        "original_component_id": item["terminal"]["original_component_id"],
        "variables": item["graph"].number_of_nodes(),
        "couplers": item["graph"].number_of_edges(),
        "max_effective_reward": item["qubo"]["max_effective_reward"],
        "penalty": item["qubo"]["penalty"],
        "penalty_margin": item["qubo"]["penalty"] - item["qubo"]["max_effective_reward"],
    }
    for item in qubo_records
])
qubo_terms = pd.DataFrame(term_rows)

qubo_summary.to_csv(OUTPUT_RUN_DIR / "terminal_qubo_summary.csv", index=False)
qubo_terms.to_csv(OUTPUT_RUN_DIR / "terminal_qubo_terms.csv", index=False)
qubo_summary.head(10)

## Classical simulated annealing

In [ ]:
def selected_nodes_from_sample(sample: dict[str, int]) -> set[str]:
    """Slots the annealer set to 1, i.e. the ones it scheduled."""
    return {node for node, value in sample.items() if int(value) == 1}


def is_independent(graph: nx.Graph, selected: set[str]) -> bool:
    """Feasibility check: no two selected slots may conflict (penalty high enough)."""
    return all(not graph.has_edge(u, v) for u, v in combinations(selected, 2))


def sample_objective(graph: nx.Graph, selected: set[str]) -> float:
    return sum(float(graph.nodes[node]["weight"]) for node in selected)


def choose_best_feasible_read(response: Any, graph: nx.Graph, qubo: dict[str, Any]) -> dict[str, Any]:
    """Prefer the best feasible annealing read; fall back to the lowest-energy read if needed."""
    reads = []
    for row in response.data(["sample", "energy"]):
        sample = {variable: int(row.sample.get(variable, 0)) for variable in qubo["variables"]}
        selected = selected_nodes_from_sample(sample)
        reads.append({
            "sample": sample,
            "selected_nodes": selected,
            "objective": sample_objective(graph, selected),
            "energy": float(row.energy),
            "is_independent": is_independent(graph, selected),
        })

    feasible_reads = [read for read in reads if read["is_independent"]]
    if feasible_reads:
        return max(feasible_reads, key=lambda read: (read["objective"], -read["energy"], tuple(sorted(read["selected_nodes"]))))
    return min(reads, key=lambda read: (read["energy"], -read["objective"]))


try:
    from dwave.samplers import SimulatedAnnealingSampler
except ImportError as exc:
    raise ModuleNotFoundError(
        "D-Wave simulated annealing is required for this notebook. "
        "Install `dwave-samplers` in the Python 3.13.14 environment, "
        "then restart the kernel and rerun the notebook."
    ) from exc

annealing_rows = []
sampler = SimulatedAnnealingSampler()   # classical stand-in for the QPU, same QUBO interface
for index, item in enumerate(qubo_records):
    qubo = item["qubo"]
    graph = item["graph"]
    terminal_id = item["terminal"]["terminal_id"]

    # The sampler wants one dict: diagonal keys (i, i) carry the linear terms.
    Q = {(node, node): coef for node, coef in qubo["linear"].items()}
    Q.update(qubo["quadratic"])

    # simulate quantum annealing
    response = sampler.sample_qubo(Q, num_reads=NUM_READS, seed=BASE_SEED + index)
    best = choose_best_feasible_read(response, graph, qubo)

    # convert the output into interpretable solution
    sample = best["sample"]
    selected = best["selected_nodes"]
    annealing_rows.append({
        "terminal_id": terminal_id,
        "selected_node_count": len(selected),
        "selected_nodes": json.dumps(sorted(selected)),
        "objective": best["objective"],
        "energy": best["energy"],
        "lowest_energy": float(response.first.energy),
        "recomputed_energy": qubo_energy(qubo, sample),
        "is_independent": best["is_independent"],
        "num_reads": NUM_READS,
        "seed": BASE_SEED + index,
    })

pd.DataFrame(annealing_rows).to_csv(OUTPUT_RUN_DIR / "annealing_results.csv", index=False)
print({"annealed_terminals": len(annealing_rows), "output": str(OUTPUT_RUN_DIR / "annealing_results.csv")})

## Reconstruct the MWIS schedule

This section walks the saved decomposition tree, combines separator branches and independent pieces, and includes forced selections. Any heuristic fallback remains explicitly identified in the output. The result is one experiment-level schedule and summary.

If any decomposition node used the greedy fallback, it is kept separate in the outputs with its feasible objective and conservative bound.

In [ ]:
# Prefer the compressed artifact; keep support for older runs with plain JSON.
RECONSTRUCTION_METADATA_GZIP = INPUT_RUN_DIR / "reconstruction_metadata.json.gz"
RECONSTRUCTION_METADATA_JSON = INPUT_RUN_DIR / "reconstruction_metadata.json"
if RECONSTRUCTION_METADATA_GZIP.exists():
    with gzip.open(RECONSTRUCTION_METADATA_GZIP, "rt", encoding="utf-8") as metadata_file:
        reconstruction_metadata = json.load(metadata_file)
elif RECONSTRUCTION_METADATA_JSON.exists():
    reconstruction_metadata = json.loads(RECONSTRUCTION_METADATA_JSON.read_text(encoding="utf-8"))
else:
    raise FileNotFoundError(
        f"No reconstruction_metadata.json[.gz] in {INPUT_RUN_DIR}. "
        "Run the MWIS pipeline first, then point qubo_input_run to that MWIS run."
    )
if reconstruction_metadata.get("producer") != "mwis_pipeline":
    raise ValueError("This reconstruction section currently supports MWIS producer runs only.")

producer_summary_path = INPUT_RUN_DIR / "final_summary.csv"
if not producer_summary_path.exists():
    raise FileNotFoundError(f"MWIS producer summary not found: {producer_summary_path}")
producer_summary = pd.read_csv(producer_summary_path)
producer_summary_values = dict(zip(producer_summary["metric"], producer_summary["value"]))
producer_upper_bound = float(producer_summary_values["Total conservative upper bound"])
producer_decomposition = pd.read_csv(INPUT_RUN_DIR / "decomposition_summary.csv")
incomplete_node_count = int(producer_decomposition["incomplete_nodes"].fillna(0).astype(int).sum())
incomplete_component_count = int((producer_decomposition["incomplete_nodes"].fillna(0).astype(int) > 0).sum())
decomposition_by_component = {
    row["component_id"]: {
        "status": row["status"],
        "incomplete_nodes": int(row["incomplete_nodes"]) if pd.notna(row["incomplete_nodes"]) else 0,
    }
    for row in producer_decomposition.to_dict(orient="records")
}

node_by_id = {node["node_id"]: node for node in reconstruction_metadata["nodes"]}
terminal_by_id = {terminal["terminal_id"]: terminal for terminal in terminal_subgraphs}
annealing_by_terminal = {row["terminal_id"]: row for row in annealing_rows}


def selected_from_row(row: dict[str, Any]) -> set[str]:
    return set(json.loads(row["selected_nodes"]))


def combine_status(statuses: list[str]) -> str:
    if any(status == "annealed_infeasible" for status in statuses):
        return "annealed_infeasible"
    if any(status == "heuristic_feasible" for status in statuses):
        return "annealed_with_heuristic_fallback"
    return "annealed_feasible"


def evaluate_reconstruction_node(node_id: str, memo: dict[str, dict[str, Any]] | None = None) -> dict[str, Any]:
    """Rebuild a feasible schedule by combining saved terminal and fallback choices.

    Memoization avoids evaluating the same shared decomposition node twice.
    """
    if memo is None:
        memo = {}
    if node_id in memo:
        return memo[node_id]

    node = node_by_id[node_id]
    base_selected = set(node.get("forced_selected", []))
    base_objective = float(node.get("forced_objective", 0.0))

    if node["node_type"] == "terminal":
        terminal_id = node.get("terminal_id")
        if terminal_id is None:
            result = {
                "status": "annealed_feasible",
                "selected_nodes": base_selected,
                "objective": base_objective,
                "terminal_ids": set(),
                "heuristic_ids": set(),
                "heuristic_objective": 0.0,
                "heuristic_upper_bound": 0.0,
                "heuristic_details": [],
            }
        else:
            row = annealing_by_terminal[terminal_id]
            terminal_record = terminal_by_id[terminal_id]
            forced_selected = set(terminal_record.get("forced_selected", []))
            selected = forced_selected | selected_from_row(row)
            result = {
                "status": "annealed_feasible" if row["is_independent"] else "annealed_infeasible",
                "selected_nodes": selected,
                "objective": float(terminal_record.get("forced_objective", 0.0)) + float(row["objective"]),
                "terminal_ids": {terminal_id},
                "heuristic_ids": set(),
                "heuristic_objective": 0.0,
                "heuristic_upper_bound": 0.0,
                "heuristic_details": [],
            }
        memo[node_id] = result
        return result

    if node["node_type"] == "incomplete":
        heuristic = node["heuristic_solution"]
        detail = {
            "node_id": node_id,
            "component_id": node["component_id"],
            "reason": node.get("reason", "incomplete decomposition"),
            "node_count": node["qpu_node_count"],
            "edge_count": node["qpu_edge_count"],
            "selected_contacts": len(heuristic["selected_nodes"]),
            "objective": float(heuristic["objective"]),
            "upper_bound": float(heuristic["upper_bound"]),
            "gap_bound": float(heuristic["upper_bound"]) - float(heuristic["objective"]),
            "selected_nodes": json.dumps(sorted(heuristic["selected_nodes"])),
        }
        result = {
            "status": "heuristic_feasible",
            "selected_nodes": set(heuristic["selected_nodes"]),
            "objective": float(heuristic["objective"]),
            "terminal_ids": set(),
            "heuristic_ids": {node_id},
            "heuristic_objective": float(heuristic["objective"]),
            "heuristic_upper_bound": float(heuristic["upper_bound"]),
            "heuristic_details": [detail],
        }
        memo[node_id] = result
        return result

    if node["node_type"] == "independent_sum":
        children = [evaluate_reconstruction_node(child_id, memo) for child_id in node["children"]]
        result = {
            "status": combine_status([child["status"] for child in children]),
            "selected_nodes": base_selected | set().union(*(child["selected_nodes"] for child in children)),
            "objective": base_objective + sum(child["objective"] for child in children),
            "terminal_ids": set().union(*(child["terminal_ids"] for child in children)),
            "heuristic_ids": set().union(*(child["heuristic_ids"] for child in children)),
            "heuristic_objective": sum(child.get("heuristic_objective", 0.0) for child in children),
            "heuristic_upper_bound": sum(child.get("heuristic_upper_bound", 0.0) for child in children),
            "heuristic_details": [detail for child in children for detail in child.get("heuristic_details", [])],
        }
        memo[node_id] = result
        return result

    branch_results = []
    for branch in node["branches"]:
        children = [evaluate_reconstruction_node(child_id, memo) for child_id in branch["children"]]
        selected = base_selected | set(branch["selected_separator"])
        for child in children:
            selected.update(child["selected_nodes"])
        branch_results.append({
            "status": combine_status([child["status"] for child in children]),
            "selected_nodes": selected,
            "objective": base_objective + float(branch["state_objective"]) + sum(child["objective"] for child in children),
            "terminal_ids": set().union(*(child["terminal_ids"] for child in children)) if children else set(),
            "heuristic_ids": set().union(*(child["heuristic_ids"] for child in children)) if children else set(),
            "heuristic_objective": sum(child.get("heuristic_objective", 0.0) for child in children),
            "heuristic_upper_bound": sum(child.get("heuristic_upper_bound", 0.0) for child in children),
            "heuristic_details": [detail for child in children for detail in child.get("heuristic_details", [])],
            "branch_state_index": branch["state_index"],
        })

    best = max(branch_results, key=lambda item: (item["objective"], tuple(sorted(item["selected_nodes"]))))
    memo[node_id] = best
    return best


component_rows = []
component_results = []
reconstruction_memo: dict[str, dict[str, Any]] = {}
for component in reconstruction_metadata["components"]:
    result = evaluate_reconstruction_node(component["root_node_id"], reconstruction_memo)
    component_results.append(result)
    decomposition = decomposition_by_component.get(component["component_id"], {})
    component_rows.append({
        "component_id": component["component_id"],
        "status": result["status"],
        "selected_contacts": len(result["selected_nodes"]),
        "objective": result["objective"],
        "terminal_subgraphs_used": len(result["terminal_ids"]),
        "heuristic_fallback_nodes": len(result["heuristic_ids"]),
        "heuristic_objective": result.get("heuristic_objective", 0.0),
        "heuristic_upper_bound": result.get("heuristic_upper_bound", 0.0),
        "decomposition_status": decomposition.get("status", "unknown"),
        "incomplete_decomposition_nodes_all_states": decomposition.get("incomplete_nodes", 0),
    })

selected_nodes = set().union(*(result["selected_nodes"] for result in component_results))
total_objective = sum(result["objective"] for result in component_results)
heuristic_ids = set().union(*(result["heuristic_ids"] for result in component_results))
terminal_ids_used = set().union(*(result["terminal_ids"] for result in component_results))
infeasible_terminals = [row for row in annealing_rows if not row["is_independent"]]
heuristic_details = [detail for result in component_results for detail in result.get("heuristic_details", [])]
heuristic_objective = sum(result.get("heuristic_objective", 0.0) for result in component_results)
heuristic_upper_bound = sum(result.get("heuristic_upper_bound", 0.0) for result in component_results)
heuristic_gap_bound = max(0.0, heuristic_upper_bound - heuristic_objective)
annealed_gap_bound = producer_upper_bound - total_objective
if annealed_gap_bound < -1e-6:
    raise ValueError("Annealed objective exceeds the MWIS producer upper bound; check input/run matching.")
annealed_gap_bound = max(0.0, annealed_gap_bound)
annealed_relative_gap_percent = 0.0 if producer_upper_bound == 0 else 100 * annealed_gap_bound / producer_upper_bound

producer_config_path = INPUT_RUN_DIR / "run_config_log.json"
producer_config = json.loads(producer_config_path.read_text(encoding="utf-8")) if producer_config_path.exists() else run_config
slot_csv = PROJECT_ROOT / producer_config["candidate_slot_csv"]
slot_df = pd.read_csv(slot_csv)
reconstructed_schedule = slot_df[slot_df["slot_id"].isin(selected_nodes)].copy()

component_provenance = pd.DataFrame(component_rows)
heuristic_fallback_details = pd.DataFrame(
    heuristic_details,
    columns=[
        "node_id",
        "component_id",
        "reason",
        "node_count",
        "edge_count",
        "selected_contacts",
        "objective",
        "upper_bound",
        "gap_bound",
        "selected_nodes",
    ],
)
summary_rows = [
    ("Producer run", INPUT_RUN_DIR.name),
    ("Status", combine_status([result["status"] for result in component_results])),
    ("Selected contacts", len(selected_nodes)),
    ("Total feasible objective", total_objective),
    ("Annealed terminal subgraphs available", len(annealing_rows)),
    ("Annealed terminal subgraphs used", len(terminal_ids_used)),
    ("Infeasible terminal annealing samples", len(infeasible_terminals)),
    ("Fallback nodes used in selected schedule", len(heuristic_ids)),
    ("Selected-schedule fallback feasible objective", heuristic_objective),
    ("Selected-schedule fallback upper bound", heuristic_upper_bound),
    ("Selected-schedule fallback gap bound", heuristic_gap_bound),
    ("Components with incomplete separator decomposition (all states)", incomplete_component_count),
    ("Incomplete separator nodes (all states)", incomplete_node_count),
    ("MWIS producer conservative objective upper bound", producer_upper_bound),
    ("Annealed schedule absolute gap bound", annealed_gap_bound),
    ("Annealed schedule relative gap bound (%)", annealed_relative_gap_percent),
    ("Certification", "annealed feasible objective matches MWIS upper bound" if annealed_gap_bound <= 1e-9 else "feasible annealed MWIS schedule; optimality not certified"),
]
reconstructed_summary = pd.DataFrame(summary_rows, columns=["metric", "value"])

reconstructed_schedule.to_csv(OUTPUT_RUN_DIR / "reconstructed_schedule.csv", index=False)
component_provenance.to_csv(OUTPUT_RUN_DIR / "component_provenance_summary.csv", index=False)
heuristic_fallback_details.to_csv(OUTPUT_RUN_DIR / "heuristic_fallback_details.csv", index=False)
reconstructed_summary.to_csv(OUTPUT_RUN_DIR / "reconstructed_summary.csv", index=False)
(OUTPUT_RUN_DIR / "reconstructed_selected_nodes.json").write_text(
    json.dumps({"selected_nodes": sorted(selected_nodes)}, indent=2),
    encoding="utf-8",
)

print({
    "selected_contacts": len(selected_nodes),
    "total_objective": total_objective,
    "infeasible_terminal_samples": len(infeasible_terminals),
    "heuristic_fallback_nodes": len(heuristic_ids),
    "heuristic_gap_bound": heuristic_gap_bound,
    "output": str(OUTPUT_RUN_DIR),
})
reconstructed_summary